# Insurance Fraud Detection - Google Colab
## Fixed for scikit-learn version compatibility

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shahidarmaan594/insurance-fraud-detection-using-ml/blob/main/colab_deployment.ipynb)

## Step 1: Clone Repository

In [ ]:
!git clone https://github.com/Shahidarmaan594/insurance-fraud-detection-using-ml.git
%cd insurance-fraud-detection-using-ml
!ls -la

## Step 2: Install Dependencies (EXACT VERSION)

In [ ]:
# ⚠️ CRITICAL: Install exact scikit-learn version used for training
!pip install -U scikit-learn==1.8.0
!pip install -q joblib pandas numpy flask flask-cors pyngrok

import sklearn
print(f"✅ scikit-learn version: {sklearn.__version__}")
print("✅ All dependencies installed!")

## Step 3: Load Pre-trained Model (with error handling)

In [ ]:
import joblib
import pickle
import sys

model_path = 'models/fraud_model.pkl'

try:
    # Try joblib first (recommended)
    print("🔄 Attempting to load model with joblib...")
    model = joblib.load(model_path)
    print(f"✅ Model loaded successfully with joblib!")
except Exception as e:
    print(f"⚠️ joblib failed: {e}")
    try:
        print("🔄 Attempting to load model with pickle...")
        with open(model_path, 'rb') as f:
            model = pickle.load(f)
        print(f"✅ Model loaded successfully with pickle!")
    except Exception as e2:
        print(f"❌ Both joblib and pickle failed!")
        print(f"Error: {e2}")
        print(f"\n⚠️ MODEL FILE MIGHT BE CORRUPTED")
        print(f"Please re-upload the model file from your local machine")
        sys.exit(1)

print(f"\n📊 Model Type: {type(model)}")
print(f"✅ Ready to make predictions!")

## Step 4: Test Model with Sample Data

In [ ]:
import pandas as pd
import numpy as np

# Create sample test data (adjust features based on your model)
sample_data = pd.DataFrame({
    'age': [35, 45, 25],
    'policy_years': [5, 10, 1],
    'claim_amount': [5000, 2000, 50000],
    'claim_frequency': [2, 1, 5],
})

print("🧪 Testing model with sample data...\n")

try:
    # Make predictions
    predictions = model.predict(sample_data)
    probabilities = model.predict_proba(sample_data)
    
    # Display results
    results_df = pd.DataFrame({
        'Prediction': ['FRAUD' if p == 1 else 'LEGITIMATE' for p in predictions],
        'Confidence': [f"{prob[1]:.2%}" for prob in probabilities]
    })
    
    print(results_df)
    print("\n✅ Model predictions working!")
    
except Exception as e:
    print(f"❌ Prediction failed: {e}")
    print(f"This might mean the model features don't match your input data")

## Step 5: Flask API Setup

In [ ]:
from flask import Flask, jsonify, request
from flask_cors import CORS
import json
import warnings
warnings.filterwarnings('ignore')

# Create Flask app
app = Flask(__name__)
CORS(app)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        data = request.json
        df = pd.DataFrame([data])
        
        prediction = model.predict(df)
        probability = model.predict_proba(df)
        
        return jsonify({
            'fraud': int(prediction[0]),
            'confidence': float(probability[0][1]),
            'label': 'FRAUD' if prediction[0] == 1 else 'LEGITIMATE'
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 400

@app.route('/health', methods=['GET'])
def health():
    return jsonify({'status': 'healthy', 'model': 'ready', 'sklearn_version': '1.8.0'})

print("✅ Flask API configured!")

## Step 6: Run API on Colab with ngrok

In [ ]:
from pyngrok import ngrok
from threading import Thread
import time

# Start Flask in background
def run_app():
    app.run(port=5000, debug=False, use_reloader=False)

thread = Thread(target=run_app, daemon=True)
thread.start()
time.sleep(3)

# Create public URL with ngrok (no auth token needed for demo)
try:
    public_url = ngrok.connect(5000)
    print(f"\n{'='*60}")
    print(f"✅ Your API is now LIVE!")
    print(f"{'='*60}")
    print(f"\n🌐 Public API URL:\n   {public_url}")
    print(f"\n📍 Endpoints:")
    print(f"   POST {public_url}/predict")
    print(f"   GET  {public_url}/health")
    print(f"\n{'='*60}\n")
except Exception as e:
    print(f"❌ Failed to create ngrok tunnel: {e}")

## Step 7: Test API Endpoint

In [ ]:
import requests
import json

# Get the ngrok URL from output above
# Then test it here

test_cases = [
    {'name': '✅ Legitimate Claim', 'data': {'age': 45, 'policy_years': 10, 'claim_amount': 2000, 'claim_frequency': 1}},
    {'name': '⚠️ Suspicious Claim', 'data': {'age': 25, 'policy_years': 1, 'claim_amount': 50000, 'claim_frequency': 5}},
    {'name': '🔴 High Risk Claim', 'data': {'age': 20, 'policy_years': 0, 'claim_amount': 100000, 'claim_frequency': 10}},
]

print("🧪 Testing API Endpoints:\n")

# Test health endpoint
try:
    health_response = requests.get(f'{public_url}/health')
    print(f"💚 Health Check: {health_response.json()}\n")
except:
    print("⚠️ Health check failed - API might still be starting...\n")

# Test predictions
for test in test_cases:
    try:
        response = requests.post(f'{public_url}/predict', json=test['data'])
        result = response.json()
        print(f"{test['name']}")
        print(f"  Result: {result['label']}")
        print(f"  Confidence: {result['confidence']:.2%}")
        print()
    except Exception as e:
        print(f"{test['name']}: ❌ Failed - {e}\n")

## Step 8: Keep API Running

In [ ]:
import time

print(f"\n{'='*60}")
print("🚀 API IS RUNNING!")
print(f"{'='*60}")
print(f"\n📎 Your Public URL: {public_url}")
print(f"\n✅ Share this link with your professor!")
print(f"\n⏰ The Colab runtime will keep this running as long as:")
print(f"   1. This cell is active (not stopped)")
print(f"   2. Browser tab remains open")
print(f"   3. No inactivity >30 minutes")
print(f"\n❌ To stop: Click the stop button or close this cell\n")
print(f"{'='*60}\n")

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\n🛑 API stopped")